# Задача 10

## Исходные данные

При 100 независимых измерениях получены частоты десятых долей:

| Цифра | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|-------|---|---|---|---|---|---|---|---|---|---|
| $m_i$ | 5 | 8 | 6 |12 |14 |18 |11 | 6 |13 | 7 |

Объём выборки $n = 100$.

---

## a) Проверка гипотезы о равномерном распределении

$H_0$: все цифры равновероятны, $p_i = 0.1$.

### 1. Критерий $\chi^2$

Ожидаемые частоты $np_i = 10$. Статистика:
$$
\chi^2 = \sum_{i=0}^{9} \frac{(m_i-10)^2}{10} = 16.4.
$$
Число степеней свободы $\text{df} = 9$.  
P‑значение: $P(\chi^2(9) \ge 16.4) \approx 0.059$.  
При $\alpha=0.05$ гипотеза не отвергается.

### 2. Критерий Колмогорова

Эмпирическая функция распределения $F_n(x)$ в точках $x=0,\dots,9$ и теоретическая $F_0(x)=\frac{x+1}{10}$:

| $x$ | $F_n(x)$ | $F_0(x)$ | $|F_n-F_0|$ |
|-----|----------|----------|------------|
| 0   | 0.05     | 0.1      | 0.05       |
| 1   | 0.13     | 0.2      | 0.07       |
| 2   | 0.19     | 0.3      | 0.11       |
| 3   | 0.31     | 0.4      | 0.09       |
| 4   | 0.45     | 0.5      | 0.05       |
| 5   | 0.63     | 0.6      | 0.03       |
| 6   | 0.74     | 0.7      | 0.04       |
| 7   | 0.80     | 0.8      | 0.00       |
| 8   | 0.93     | 0.9      | 0.03       |
| 9   | 1.00     | 1.0      | 0.00       |

Максимальное отклонение $D_n = 0.11$.  
Статистика $\tilde{\Delta} = \sqrt{n}\,D_n = 10 \cdot 0.11 = 1.1$.  
P‑значение для критерия Колмогорова (точное):
$$
\text{P-value} = 1 - K(1.1) = 2\sum_{k=1}^\infty (-1)^{k-1} e^{-2k^2\cdot 1.1^2} \approx 0.03968.
$$
Так как $0.03968 < 0.05$, гипотеза $H_0$ отвергается.  
Критерий Колмогорова оказался более чувствительным и обнаружил систематическое отклонение, тогда как $\chi^2$ его не уловил.

---

## b) Проверка гипотезы о нормальном распределении

$H_0$: данные подчиняются нормальному закону с неизвестными параметрами $\mu$ и $\sigma$.

### 1. Оценка параметров

Выборочное среднее:
$$
\bar{x} = \frac{1}{100}\sum_{i=0}^{9} i\,m_i = 4.77.
$$
Несмещённая выборочная дисперсия:
$$
s^2 = \frac{1}{99}\sum_{i=0}^{9} m_i (i - \bar{x})^2 \approx 6.341, \quad s \approx 2.518.
$$

### 2. Критерий $\chi^2$

---

| Интервал       | $(-\infty,1)$ | $[1,2)$ | $[2,3)$ | $[3,4)$ | $[4,5)$ | $[5,6)$ | $[6,7)$ | $[7,8)$ | $[8,9)$ | $(9,\infty)$ |
|----------------|---------------|---------|---------|---------|---------|---------|---------|---------|---------|--------------|
| $nP_i$         | 6.72          | 6.85    | 10.54   | 13.88   | 15.65   | 15.10   | 12.47   | 8.81    | 5.33    | 4.65         |

---

Статистика:
$$
\chi^2 = \sum_{i=0}^{9} \frac{(m_i - nP_i)^2}{nP_i} \approx 16.871.
$$
Число степеней свободы: $\text{df} = 10 - 1 - 2 = 7$ (оценены два параметра).  
P‑значение: $P(\chi^2(7) \ge 16.871) \approx 0.01825$.  
При $\alpha=0.05$ гипотеза $H_0$ отвергается.

### 3. Критерий Колмогорова (с учётом оценки параметров – параметрический бутстрап)

Поскольку параметры оцениваются по выборке, распределение статистики Колмогорова зависит от оцениваемых параметров. Для получения p‑значения применяем параметрический бутстрап.

Алгоритм:
1. По исходной выборке находим оценки $\hat\mu$, $\hat\sigma$.
2. Вычисляем $D_n$ для исходных данных.
3. Генерируем $M$ выборок объёмом $n$ из $N(\hat\mu,\hat\sigma^2)$.
4. Для каждой бутстрап‑выборки:
   - находим оценки $\mu^*$, $\sigma^*$,
   - вычисляем $D_n^*$ как максимальное отклонение эмпирической функции распределения бутстрап‑выборки от теоретической $N(\mu^*,\sigma^{*2})$,
   - получаем $\Delta^* = \sqrt{n} D_n^*$.
5. Исходная статистика $\tilde{\Delta} = \sqrt{n} D_n$.
6. P‑значение = доля $\Delta^* \ge \tilde{\Delta}$.
.

In [ ]:
import numpy as np
import math
from scipy.stats import chi2, kstest, norm
from scipy.special import erf

# Данные
m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
N = 100
alpha = 0.05

# Векторизованная функция нормального распределения
def F_norm(x, mu, sigma):
    return 0.5 * (1 + erf((x - mu) / (np.sqrt(2) * sigma)))

print("="*50)
print("Задача 10")
print("="*50)

# ============================================================
# a) Равномерное распределение
# ============================================================
print("\n=== a) Равномерное распределение ===\n")

# χ²
expected_unif = np.full_like(m, N/10)
chi2_unif = np.sum((m - expected_unif)**2 / expected_unif)
df_unif = 9
p_unif_chi2 = 1 - chi2.cdf(chi2_unif, df_unif)
print(f"χ² = {chi2_unif:.4f}, df = {df_unif}, p-value = {p_unif_chi2:.6f}")

# Колмогоров (kstest для равномерного распределения на [0,10])
# Для непрерывной аппроксимации сгенерируем выборку из повторяющихся значений
sample = np.repeat(np.arange(10), m)
ks_result = kstest(sample, 'uniform', args=(0, 10))
p_unif_kolmogorov = ks_result.pvalue
print(f"Колмогоров (kstest): D_n = {ks_result.statistic:.4f}, p-value = {p_unif_kolmogorov:.6f}")

if p_unif_chi2 < alpha:
    print("χ²: H0 отвергается")
else:
    print("χ²: H0 не отвергается")

if p_unif_kolmogorov < alpha:
    print("Колмогоров: H0 отвергается")
else:
    print("Колмогоров: H0 не отвергается")

print("\nВывод: критерий Колмогорова отвергает равномерность, χ² не отвергает.")

# ============================================================
# b) Нормальное распределение
# ============================================================
print("\n=== b) Нормальное распределение ===\n")

# Оценка параметров
mu_hat = np.mean(sample)
sigma_hat = np.std(sample, ddof=1)
print(f"Оценки: μ = {mu_hat:.4f}, σ = {sigma_hat:.4f}")

# χ² с 10 интервалами
segments = [(-np.inf, 1)] + [(i, i+1) for i in range(1, 9)] + [(9, np.inf)]
probs = [norm.cdf(seg[1], mu_hat, sigma_hat) - norm.cdf(seg[0], mu_hat, sigma_hat) for seg in segments]
exp_norm = N * np.array(probs)

print("\nОжидаемые и наблюдаемые частоты:")
print("i    | m_i   | nP_i")
print("-" * 25)
for i in range(10):
    print(f"{i:2d}   | {m[i]:3d}   | {exp_norm[i]:6.2f}")

chi2_norm = np.sum((m - exp_norm)**2 / exp_norm)
df_norm = 10 - 1 - 2  # 7 степеней свободы
p_norm_chi2 = 1 - chi2.cdf(chi2_norm, df_norm)
print(f"\nχ² = {chi2_norm:.4f}, df = {df_norm}, p-value = {p_norm_chi2:.6f}")

# Колмогоров с параметрическим бутстрапом (как в ноутбуке)
print("\nПараметрический бутстрап для критерия Колмогорова...")

F_emp_cum = np.cumsum(m) / N
x_vals = np.arange(10)
F_theor = F_norm(x_vals, mu_hat, sigma_hat)
D_orig = np.max(np.abs(F_emp_cum - F_theor))
delta_orig = np.sqrt(N) * D_orig
print(f"Исходная статистика: D_n = {D_orig:.4f}, Δ = {delta_orig:.4f}")

M = 50000
boot_deltas = np.zeros(M)
for i in range(M):
    boot_sample = np.random.normal(mu_hat, sigma_hat, N)
    mu_boot = np.mean(boot_sample)
    sigma_boot = np.std(boot_sample, ddof=1)
    boot_vals = np.sort(boot_sample)
    F_boot_emp = np.arange(1, N+1) / N
    F_boot_theor = F_norm(boot_vals, mu_boot, sigma_boot)
    D_boot = np.max(np.abs(F_boot_emp - F_boot_theor))
    boot_deltas[i] = np.sqrt(N) * D_boot

p_norm_boot = np.mean(boot_deltas >= delta_orig)
print(f"Колмогоров (бутстрап): p-value = {p_norm_boot:.6f}")

if p_norm_chi2 < alpha:
    print("χ²: H0 отвергается")
else:
    print("χ²: H0 не отвергается")

if p_norm_boot < alpha:
    print("Колмогоров (бутстрап): H0 отвергается")
else:
    print("Колмогоров (бутстрап): H0 не отвергается")

print("\nВывод: оба критерия отвергают гипотезу о нормальности.")

Задача 10

=== a) Равномерное распределение ===

χ² = 16.4000, df = 9, p-value = 0.058984
Колмогоров (kstest): D_n = 0.1400, p-value = 0.035825
χ²: H0 не отвергается
Колмогоров: H0 отвергается

Вывод: критерий Колмогорова отвергает равномерность, χ² не отвергает.

=== b) Нормальное распределение ===

Оценки: μ = 4.7700, σ = 2.5180

Ожидаемые и наблюдаемые частоты:
i    | m_i   | nP_i
-------------------------
 0   |   5   |   6.72
 1   |   8   |   6.85
 2   |   6   |  10.54
 3   |  12   |  13.88
 4   |  14   |  15.65
 5   |  18   |  15.10
 6   |  11   |  12.47
 7   |   6   |   8.81
 8   |  13   |   5.33
 9   |   7   |   4.65

χ² = 16.8711, df = 7, p-value = 0.018247

Параметрический бутстрап для критерия Колмогорова...
Исходная статистика: D_n = 0.0936, Δ = 0.9361
Колмогоров (бутстрап): p-value = 0.020460
χ²: H0 отвергается
Колмогоров (бутстрап): H0 отвергается

Вывод: оба критерия отвергают гипотезу о нормальности.
